In [1]:
import pandas as pd

In [4]:
raw = pd.read_csv(r"C:\Users\Denzen Computer\Downloads\teaching_prices_messy (1).csv")

In [13]:
raw

,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
...,...,...,...,...,...,...
92,2026-03-19,Pulse C,City 1,207.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
93,2026-03-19,Pulse C,City 2,213.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
94,2026-03-19,Sweetener D,City 1,178.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
95,2026-03-19,Sweetener D,City 2,184.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


In [5]:
clean = raw.copy()

In [6]:
clean['item'] = clean['item'].str.strip().str.title()

In [8]:
clean['price'] = clean['price'].astype(str).str.replace(',', '', regex=False)

In [9]:
clean['price'] = pd.to_numeric(clean['price'], errors='coerce')

In [10]:
clean.loc[clean['price'] == 9999, 'price'] = float('nan')

In [11]:
clean = clean.drop_duplicates().reset_index(drop=True)


In [12]:
print(clean.shape)

(96, 6)


In [14]:
print(clean['price'].dtype)

float64


In [15]:
pivot = clean.pivot_table(index='date', columns='item', values='price', aggfunc='mean')

In [16]:
pivot = pivot.sort_index()

In [17]:
pivot

item,Oil B,Pulse C,Staple A,Sweetener D
date,,,,
2026-01-01,301.000000,200.0,104.0,152.0
2026-01-08,307.000000,205.5,102.5,156.5
2026-01-15,313.000000,203.0,676.5,156.0
2026-01-22,314.000000,205.5,109.5,163.5
2026-01-29,320.000000,203.0,108.0,165.0
2026-02-05,250.166667,205.5,114.5,164.5
2026-02-12,327.000000,208.0,110.0,169.0
2026-02-19,333.000000,205.5,113.5,168.5
2026-02-26,334.000000,208.0,117.0,173.0


In [18]:
stats = pd.DataFrame()

In [19]:
stats['mean'] = pivot.mean()

In [20]:
stats['median'] = pivot.median()

In [21]:
stats['first_price'] = pivot.iloc[0]

In [22]:
stats['last_price'] = pivot.iloc[-1]

In [23]:
stats['pct_change'] = (stats['last_price'] - stats['first_price']) / stats['first_price'] * 100

In [24]:
stats.round(2)

,mean,median,first_price,last_price,pct_change
item,,,,,
Oil B,318.93,323.50,301.0,347.0,15.28
Pulse C,205.67,205.50,200.0,210.5,5.25
Staple A,158.96,114.00,104.0,117.5,12.98
Sweetener D,167.00,166.75,152.0,181.5,19.41


In [25]:
staple = clean.loc[clean['item'] == 'Staple A', 'price'].dropna()

In [26]:
normal = staple[staple < 1000]

In [27]:
flagged = staple[staple >= 1000]

In [29]:
print('WITH flag    : mean = %.2f , median = %.2f , n = %d' % (staple.mean(), staple.median(), len(staple)))

WITH flag    : mean = 160.89 , median = 112.50 , n = 23


In [30]:
print('WITHOUT flag : mean = %.2f , median = %.2f , n = %d' % (normal.mean(), normal.median(), len(normal)))

WITHOUT flag : mean = 111.39 , median = 112.50 , n = 22


In [31]:
print('Flagged values:', flagged.tolist())

Flagged values: [1250.0]


In [32]:
sweet = clean.loc[clean['item'] == 'Sweetener D', 'price'].dropna()

In [33]:
sweet_no = clean.loc[(clean['item'] == 'Sweetener D') & (clean['date'] != '2026-01-22'), 'price'].dropna()

In [42]:
print('WITH flag : mean = %.2f , median = %.2f , n = %d' % (sweet.mean(), sweet.median(), len(sweet)))

WITH flag : mean = 167.15 , median = 167.50 , n = 23


In [35]:
print('WITHOUT flag : mean = %.2f , median = %.2f , n = %d' % (sweet_no.mean(), sweet_no.median(), len(sweet_no)))

WITHOUT flag : mean = 167.32 , median = 167.75 , n = 22


In [43]:
print('Staple A mean          =', round(staple.mean(), 2))      

Staple A mean          = 160.89


In [44]:
print('Staple A median        =', round(staple.median(), 2))    

Staple A median        = 112.5


In [45]:
print('Staple A mean no flag  =', round(normal.mean(), 2))      

Staple A mean no flag  = 111.39


In [46]:
print('Staple A median no flag=', round(normal.median(), 2))    

Staple A median no flag= 112.5


In [40]:
print(clean['city'].unique())

<ArrowStringArray>
['City 1', 'City 2']
Length: 2, dtype: str


In [41]:
print(clean['date'].min(), 'to', clean['date'].max())

2026-01-01 to 2026-03-19
